In [1]:
# config.py
# ==========================================================

from tvDatafeed import Interval

# ----------------------------------------------------------
# GENEL
# ----------------------------------------------------------

TIMEFRAME = Interval.in_daily    # indikatörler ile analizi ise saatlik olarak yapıyoruz.
#TIMEFRAME = '1h'
BARS = 300
SEARCH_BARS = 120

# AI Score tarama sonucu filtrelenirken kullanılacak alt limit.
# Sadece bu puan ve üzerindeki hisseler sonuç listesine (CSV) eklenir.
MIN_SCORE = 70 

# ----------------------------------------------------------
# EMA / SMA
# ----------------------------------------------------------

EMA_LIST = [5, 8, 13, 20, 34, 50, 89, 100, 144, 200]
SMA_LIST = [20, 50, 100, 200]

# ----------------------------------------------------------
# RSI
# ----------------------------------------------------------

RSI_LENGTH = 14
RSI_EMA = 8
RSI_OVERSOLD = 30
RSI_RECOVERY = 40
RSI_OVERBOUGHT = 70

# ----------------------------------------------------------
# MACD
# ----------------------------------------------------------

MACD_FAST = 12
MACD_SLOW = 26
MACD_SIGNAL = 9

# ----------------------------------------------------------
# CCI / ROC / MOMENTUM / ADX / ATR
# ----------------------------------------------------------

CCI_LENGTH = 20
ROC_LENGTH = 10
MOMENTUM_LENGTH = 10
ADX_LENGTH = 14
ADX_LIMIT = 20
ATR_LENGTH = 14

# ----------------------------------------------------------
# STOCHASTIC
# ----------------------------------------------------------

STOCH_LENGTH = 14
STOCH_K_SMOOTH = 3
STOCH_D_SMOOTH = 3

# ----------------------------------------------------------
# HACİM & DİP
# ----------------------------------------------------------

RVOL_LIMIT = 1.20
VOL_MA = 20
DIP_LOOKBACK = 80
RSI_DIP = 32

# ----------------------------------------------------------
# FIBONACCI
# ----------------------------------------------------------
FIB_LOOKBACK = 100  # Fibonacci tepe/dip analizi için geriye dönük mum sayısı

In [2]:
# indikatör.py
# Bu dosya, indikatörlerin tanımlandığı ve hesaplandığı bir Python modülüdür. 
# Bu modül, finansal veriler üzerinde çeşitli teknik analiz indikatörlerini hesaplamak için kullanılabilir.

import numpy as np
import pandas as pd
import pandas_ta as ta


def calculate_core_indicators(df):
    """
    Kategori 1 ve Kategori 2'deki yüksek getirili çekirdek indikatörleri hesaplar.
    """
    df = df.copy()

    # 1. MOMENTUM & DİP İNDİKATÖRLERİ
    df["RSI"] = ta.rsi(df["close"], length=14)

    # Stochastic (14, 3, 3)
    stoch = ta.stoch(df["high"], df["low"], df["close"], k=14, d=3, smooth_k=3)
    if stoch is not None and not stoch.empty:
        df["STOCHk"] = stoch.iloc[:, 0]
        df["STOCHd"] = stoch.iloc[:, 1]

    # Williams %R (14)
    df["WILLR"] = ta.willr(df["high"], df["low"], df["close"], length=14)

    # 2. TREND & ZAYIFLAMA (ADX / DI)
    adx_df = ta.adx(df["high"], df["low"], df["close"], length=14)
    if adx_df is not None and not adx_df.empty:
        df["ADX"] = adx_df.iloc[:, 0]
        df["DMP"] = adx_df.iloc[:, 1]  # DI+
        df["DMN"] = adx_df.iloc[:, 2]  # DI-

    # EMA'lar (Trend Devam Şartı İçin Değil, Erken Dip Kontrolü İçin)
    df["EMA20"] = ta.ema(df["close"], length=20)
    df["EMA50"] = ta.ema(df["close"], length=50)

    # 3. HACİM VE AKILLI PARA İNDİKATÖRLERİ
    df["MFI"] = ta.mfi(df["high"], df["low"], df["close"], df["volume"], length=14)
    df["OBV"] = ta.obv(df["close"], df["volume"])

    # RVOL (Relative Volume - 20 Günlük Hacim Ortalamasına Göre)
    vol_sma = ta.sma(df["volume"], length=20)
    df["RVOL"] = df["volume"] / np.where(vol_sma == 0, 1, vol_sma)

    # 4. VOLATİLİTE & BANTLAR (Bollinger Bands)
    bbands = ta.bbands(df["close"], length=20, std=2)
    if bbands is not None and not bbands.empty:
        df["BBL"] = bbands.iloc[:, 0]  # Lower Band
        df["BBM"] = bbands.iloc[:, 1]  # Middle Band
        df["BBU"] = bbands.iloc[:, 2]  # Upper Band

    # 5. ONAY İNDİKATÖRLERİ (Kategori 2)
    # MACD Histogram
    macd = ta.macd(df["close"], fast=12, slow=26, signal=9)
    if macd is not None and not macd.empty:
        df["MACDh"] = macd.iloc[:, 1]  # Histogram

    # Force Index (13)
    df["EFI"] = ta.efi(df["close"], df["volume"], length=13)

    # Chaikin Oscillator
    df["ADOSC"] = ta.adosc(
        df["high"], df["low"], df["close"], df["volume"], fast=3, slow=10
    )

    return df


In [3]:
# scanner.py
# TradingView Screener + tvDatafeed
# ==========================================================

from tradingview_screener import Query
from tvDatafeed import TvDatafeed
from config import *

# ----------------------------------------------------------
# TradingView
# ----------------------------------------------------------

tv = TvDatafeed()

# ----------------------------------------------------------
# BIST Hisse Listesi
# ----------------------------------------------------------

def get_bist_symbols():

    rows, df = (
        Query()
        .set_markets("turkey")
        .set_property("interval", "1D") # Hisse elemesini günlükte yapıyoruz
        .select(

            "name",

            "close",

            "volume",

            "RSI",

            "ADX",

            "EMA20",

            "EMA50",

            "relative_volume_10d_calc"

        )
        .limit(1000)
        .get_scanner_data()
    )

    return df


# ----------------------------------------------------------
# Ön Eleme
# ----------------------------------------------------------

def pre_filter(df):
    """Sadece teknik olarak işlem görmeyen/ölü tahtaları eler."""
    df = df.copy()

    # Sert RVOL filtresini BURADAN KALDIRDIK!
    # Sadece fiyatı ve hacmi sıfır olan (işleme kapalı) tahtaları eliyoruz.
    df = df[(df["close"] > 0) & (df["volume"] > 0)]

    return df.reset_index(drop=True)


# ----------------------------------------------------------
# Mum Verisi
# ----------------------------------------------------------

def get_history(symbol):

    try:

        data = tv.get_hist(

            symbol=symbol,

            exchange="BIST",

            interval=TIMEFRAME,

            n_bars=BARS

        )

        if data is None:
            return None

        if data.empty:
            return None

        return data

    except Exception:

        return None


you are using nologin method, data you access may be limited


In [4]:
import numpy as np
import pandas as pd


def calculate_fibonacci_dip_score(df, lookback_period=100):
    """Belirlenen periyottaki Fibonacci dip bölgelerini ve fiyata olan mesafesini puanlar."""
    if len(df) < lookback_period:
        return 0, "Yetersiz Veri"

    # Son 'lookback_period' mum verisi
    recent_df = df.iloc[-lookback_period:]

    highest_high = recent_df["high"].max()
    lowest_low = recent_df["low"].min()
    current_close = df.iloc[-1]["close"]

    diff = highest_high - lowest_low
    if diff == 0:
        return 0, "Yatay Piyasa"

    # ------------------------------------------------------
    # Doğru Fibonacci Seviyeleri (Dipten Yukarıya Yüzdelik Mesafe)
    # ------------------------------------------------------
    fib_0 = lowest_low  # %0 (Mutlak Dip)
    fib_236 = lowest_low + (diff * 0.236)  # Dipten %23.6 yukarıda
    fib_382 = lowest_low + (diff * 0.382)  # Dipten %38.2 yukarıda
    fib_500 = lowest_low + (diff * 0.500)  # Orta Nokta

    score = 0
    status = "NEUTRAL"

    # ------------------------------------------------------
    # YENİ "SADECE DİP" PUANLAMA MANTIĞI
    # ------------------------------------------------------

    # 1. MUTLAK TABAN / EN DERİN DİP BÖLGESİ (En Yüksek Puan!)
    # Fiyat en düşük dip ile dipten %23.6 tepki alanı arasındaysa (Tam Akümülasyon/Bıçak Yerde)
    if fib_0 <= current_close <= fib_236:
        score = 100
        status = "FIB_ABSOLUTE_BOTTOM"  # Tam Taban/Dip Alanı

    # 2. İLK ERKEN TEPKİ / DİPTEN HAFİF KAFASINI KALDIRAN BÖLGE
    # Fiyat dipten %23.6 ile %38.2 aralığına gelmişse
    elif fib_236 < current_close <= fib_382:
        score = 75
        status = "FIB_EARLY_REBOUND"  # Erken Tepki Alanı

    # 3. ORTA BÖLGE VEYA TEPEYE YAKIN
    # Dip alanından tamamen uzaklaşmış
    elif fib_382 < current_close <= fib_500:
        score = 40
        status = "FIB_MID_ZONE"
    else:
        score = 10
        status = "FIB_HIGH_ZONE"  # Tepeye yakın, dip değil

    return score, status

In [5]:
# dip.py 
# ==========================================================


def calculate_dip_engine_score(df):
    """Hacimsiz dip yapan hisseleri elemez, hacim gelirse ekstra ödüllendirir."""
    # Yetersiz veri varsa direkt ele
    if len(df) < 100:
        return {"score": 0, "status": "INSUFFICIENT_DATA", "passed_prefilter": False}

    last = df.iloc[-1]
    prev = df.iloc[-2]

    score = 0

    # ------------------------------------------------------
    # 1. MOMENTUM & DİP İNDİKATÖRLERİ (Ana Yük)
    # ------------------------------------------------------
    # RSI (Max 25 Puan)
    if last["RSI"] < 30:
        score += 25
    elif last["RSI"] < 35:
        score += 18
    elif last["RSI"] < 40:
        score += 10

    # MFI - Para Girişi (Max 20 Puan)
    if last["MFI"] < 25:
        score += 20
    elif last["MFI"] < 35:
        score += 12

    # OBV Akıllı Para Birikimi (Max 15 Puan)
    if last["OBV"] > prev["OBV"]:
        score += 10
        if last["close"] <= prev["close"]:  # Fiyat düşerken/yatayken OBV artıyorsa
            score += 5

    # Bollinger Bands Alt Bant (Max 15 Puan)
    if last["close"] <= last["BBL"] * 1.01:
        score += 15

    # Williams %R & Stochastic (Max 15 Puan)
    if last["WILLR"] < -80:
        score += 8
    if last["STOCHk"] < 20:
        score += 7

    # ------------------------------------------------------
    # 2. HACİM & RVOL (FİLTRE DEĞİL -> SADECE BONUS PUAN)
    # ------------------------------------------------------
    # Hacim düşükse elenmez, ama dipten hacimle kalkıyorsa +10 bonus alır!
    if last["RVOL"] >= 1.2:
        score += 10  # Hacimli kalkış bonusu
    elif last["RVOL"] >= 0.8:
        score += 5  # Normal hacim katkısı
    # RVOL < 0.8 ise 0 puan alır AMA ELENMEZ! (Sessiz toplanan dip kağıdı)

    # ------------------------------------------------------
    # 3. ONAY BONUSLARI (Max +10 Puan)
    # ------------------------------------------------------
    if last["MACDh"] > prev["MACDh"]:
        score += 5
    if last["EFI"] > 0 or last["ADOSC"] > 0:
        score += 5

    final_score = min(score, 100)

    # Status belirleme
    if final_score >= 80:
        status = "YUKSEK_OLASILIKLI_DIP"
    elif final_score >= 65:
        status = "SESSİZ_AKUMULASYON_DIP"  # Hacimsiz ama göstergeleri diplere vurmuş
    elif final_score >= 50:
        status = "IZLEME_BOLGESI"
    else:
        status = "DIP_DEGIL"

    return {"score": final_score, "status": status, "passed_prefilter": True}


def calculate_final_ai_score(dip_res, fib_score):
    """
    Dip Motor Skoru (%85) ve Fibonacci Taban Skoru'nu (%15) birleştirir.
    """
    raw_dip_score = dip_res["score"]

    # Fibonacci Katkısı
    # fib_score 0-100 arası gelir, %15 ağırlıkla eklenir.
    final_ai_score = round((raw_dip_score * 0.85) + (fib_score * 0.15), 2)

    # Sinyal Sınıflandırma
    if final_ai_score >= 80:
        signal = "AL"  # Güçlü Dip Adayı
    elif final_ai_score >= 65:
        signal = "ERKEN"  # Takibe Alınacak Dip
    elif final_ai_score >= 50:
        signal = "IZLE"
    else:
        signal = "BEKLE"

    return {
        "ai_score": final_ai_score,
        "signal": signal,
        "status": dip_res["status"],
        "passed_prefilter": dip_res["passed_prefilter"],
    }

In [6]:
# ==========================================================
# BIST ENGINE PRO v2.0 - DIP HUNTER (Jupyter Notebook Cell)
# ==========================================================

print("=" * 70)
print("              BIST ENGINE PRO v2.0 - DIP HUNTER")
print("=" * 70)

# ----------------------------------------------------------
# 1. Hisse Çekme ve Ön Eleme
# ----------------------------------------------------------
symbols_df = get_bist_symbols()

if symbols_df is None or symbols_df.empty:
    print("❌ Hisse listesi alınamadı. İşlem durduruldu.")
else:
    print(f"📊 BIST Toplam Çekilen Hisse : {len(symbols_df)}")

    # Esnek Hacim / Canlılık Filtresi
    filtered_symbols = pre_filter(symbols_df)

    results = []

    # ----------------------------------------------------------
    # 2. Analiz Döngüsü
    # ----------------------------------------------------------
    print("\n🔍 Dip Taraması Başlatılıyor...\n")

    for _, row in filtered_symbols.iterrows():
        symbol = row["name"]

        try:
            # Veri Çekme
            df = get_history(symbol)
            if df is None or len(df) < 100:
                continue

            # A. Çekirdek İndikatörleri Hesapla
            df = calculate_core_indicators(df)

            # B. Dip Motor Skorunu Hesapla (0 - 100 Puan)
            dip_res = calculate_dip_engine_score(df)

            # C. Fibonacci Taban Yakınlık Skorunu Hesapla
            fib_score, fib_status = calculate_fibonacci_dip_score(
                df, lookback_period=100
            )

            # D. Nihai AI Skorunu Oluştur
            ai_res = calculate_final_ai_score(dip_res, fib_score)

            # Anlık Akış (Konsol Bilgisi)
            print(
                f"[{symbol}] ➔ AI Score: {ai_res['ai_score']} | "
                f"Dip Skoru: {dip_res['score']} | "
                f"Fib Status: {fib_status} | "
                f"Fib Score: {fib_score} | "
                f"Sinyal: {ai_res['signal']}"
            )

            # Tablo İçin Listeye Ekle
            results.append({
                "Hisse": symbol,
                "AI Score": ai_res["ai_score"],
                "Sinyal": ai_res["signal"],
                "Dip Puanı": dip_res["score"],
                "Fib Puanı": fib_score,
                "Dip Durumu": dip_res["status"],
                "Fib Durumu": fib_status,
            })

        except Exception as e:
            print(f"⚠️ {symbol} Analiz Hatası -> {e}")

    # ----------------------------------------------------------
    # 3. Sonuçları Sırala ve Notebook Tablosu Olarak Göster
    # ----------------------------------------------------------
    if len(results):
        df_results = pd.DataFrame(results)

        # AI Score'a Göre En Yüksekten En Düşüğe Sırala
        df_results = df_results.sort_values(by="AI Score", ascending=False)

        # CSV Kaydı
        df_results.to_csv(
            "BIST_DIP_HUNTER_RESULTS.csv", index=False, encoding="utf-8-sig"
        )
        print(
            "\n✅ Sonuçlar 'BIST_DIP_HUNTER_RESULTS.csv' dosyasına kaydedildi.\n"
        )

        print("=" * 80)
        print("                 TARAMA SONUÇLARI (EN YÜKSEK DİP POTANSİYELİ)")
        print("=" * 80)

    else:
        print("\n❌ Aranan kriterlerde uygun dip hissesi bulunamadı.")
        df_results = pd.DataFrame()

# Jupyter Notebook'ta HTML Tablo Olarak Şık Görünüm (En Son Satır)
df_results.head(20)  # En yüksek AI Score alan ilk 20 hisseyi gösterir

              BIST ENGINE PRO v2.0 - DIP HUNTER
📊 BIST Toplam Çekilen Hisse : 608

🔍 Dip Taraması Başlatılıyor...

[ASELS] ➔ AI Score: 18.5 | Dip Skoru: 20 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[QNBTR] ➔ AI Score: 57.5 | Dip Skoru: 50 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: IZLE
[HEDEF] ➔ AI Score: 5.75 | Dip Skoru: 5 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[DSTKF] ➔ AI Score: 63.95 | Dip Skoru: 62 | Fib Status: FIB_EARLY_REBOUND | Sinyal: IZLE
[TUPRS] ➔ AI Score: 14.25 | Dip Skoru: 15 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[ENKAI] ➔ AI Score: 15.0 | Dip Skoru: 0 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: BEKLE
[GARAN] ➔ AI Score: 50.7 | Dip Skoru: 42 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: IZLE
[KCHOL] ➔ AI Score: 10.0 | Dip Skoru: 10 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[BIMAS] ➔ AI Score: 5.75 | Dip Skoru: 5 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[THYAO] ➔ AI Score: 31.25 | Dip Skoru: 35 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[KTLEV] ➔ AI Sco

ERROR:tvDatafeed.main:Connection to remote host was lost.
ERROR:tvDatafeed.main:no data, please check the exchange and symbol


[DMLKT] ➔ AI Score: 31.25 | Dip Skoru: 35 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[RGYAS] ➔ AI Score: 5.75 | Dip Skoru: 5 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE


ERROR:tvDatafeed.main:Connection to remote host was lost.
ERROR:tvDatafeed.main:no data, please check the exchange and symbol


[EUPWR] ➔ AI Score: 5.75 | Dip Skoru: 5 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[RYGYO] ➔ AI Score: 5.75 | Dip Skoru: 5 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[LYDHO] ➔ AI Score: 32.0 | Dip Skoru: 20 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: BEKLE
[TABGD] ➔ AI Score: 15.5 | Dip Skoru: 5 | Fib Status: FIB_EARLY_REBOUND | Sinyal: BEKLE
[AYGAZ] ➔ AI Score: 14.25 | Dip Skoru: 15 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[BIGEN] ➔ AI Score: 18.5 | Dip Skoru: 20 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[AKFIS] ➔ AI Score: 5.75 | Dip Skoru: 5 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[BSOKE] ➔ AI Score: 5.75 | Dip Skoru: 5 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE


...

ERROR:tvDatafeed.main:Connection to remote host was lost.
ERROR:tvDatafeed.main:no data, please check the exchange and symbol


[DERIM] ➔ AI Score: 45.25 | Dip Skoru: 40 | Fib Status: FIB_EARLY_REBOUND | Sinyal: BEKLE
[PENGD] ➔ AI Score: 36.75 | Dip Skoru: 30 | Fib Status: FIB_EARLY_REBOUND | Sinyal: BEKLE
[AVGYO] ➔ AI Score: 14.25 | Dip Skoru: 15 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[GLRYH] ➔ AI Score: 43.9 | Dip Skoru: 34 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: BEKLE
[RUBNS] ➔ AI Score: 32.0 | Dip Skoru: 20 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: BEKLE
[KFEIN] ➔ AI Score: 28.25 | Dip Skoru: 20 | Fib Status: FIB_EARLY_REBOUND | Sinyal: BEKLE
[DMSAS] ➔ AI Score: 14.5 | Dip Skoru: 10 | Fib Status: FIB_MID_ZONE | Sinyal: BEKLE
[TLMAN] ➔ AI Score: 66.0 | Dip Skoru: 60 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: ERKEN
[CUSAN] ➔ AI Score: 27.25 | Dip Skoru: 25 | Fib Status: FIB_MID_ZONE | Sinyal: BEKLE
[A1YEN] ➔ AI Score: 38.45 | Dip Skoru: 32 | Fib Status: FIB_EARLY_REBOUND | Sinyal: BEKLE
[MARKA] ➔ AI Score: 10.0 | Dip Skoru: 10 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[EYGYO] ➔ AI Score: 19.25 |

ERROR:tvDatafeed.main:Connection to remote host was lost.
ERROR:tvDatafeed.main:no data, please check the exchange and symbol


[SMART] ➔ AI Score: 32.0 | Dip Skoru: 20 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: BEKLE
[YONGA] ➔ AI Score: 77.05 | Dip Skoru: 73 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: ERKEN
[ETYAT] ➔ AI Score: 12.55 | Dip Skoru: 13 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[VKFYO] ➔ AI Score: 40.5 | Dip Skoru: 30 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: BEKLE
[EUKYO] ➔ AI Score: 14.25 | Dip Skoru: 15 | Fib Status: FIB_HIGH_ZONE | Sinyal: BEKLE
[KRTEK] ➔ AI Score: 76.2 | Dip Skoru: 72 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: ERKEN
[BRKSN] ➔ AI Score: 32.0 | Dip Skoru: 20 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: BEKLE
[RODRG] ➔ AI Score: 28.95 | Dip Skoru: 27 | Fib Status: FIB_MID_ZONE | Sinyal: BEKLE
[IHEVA] ➔ AI Score: 40.5 | Dip Skoru: 30 | Fib Status: FIB_ABSOLUTE_BOTTOM | Sinyal: BEKLE
[OYLUM] ➔ AI Score: 19.75 | Dip Skoru: 10 | Fib Status: FIB_EARLY_REBOUND | Sinyal: BEKLE
[IDGYO] ➔ AI Score: 31.5 | Dip Skoru: 30 | Fib Status: FIB_MID_ZONE | Sinyal: BEKLE
[SAMAT] ➔ AI Score: 57

,Hisse,AI Score,Sinyal,Dip Puanı,Fib Puanı,Dip Durumu,Fib Durumu
553,YONGA,77.05,ERKEN,73,100,SESSİZ_AKUMULASYON_DIP,FIB_ABSOLUTE_BOTTOM
557,KRTEK,76.20,ERKEN,72,100,SESSİZ_AKUMULASYON_DIP,FIB_ABSOLUTE_BOTTOM
298,KLGYO,74.50,ERKEN,70,100,SESSİZ_AKUMULASYON_DIP,FIB_ABSOLUTE_BOTTOM
161,LILAK,72.80,ERKEN,68,100,SESSİZ_AKUMULASYON_DIP,FIB_ABSOLUTE_BOTTOM
271,GIPTA,72.80,ERKEN,68,100,SESSİZ_AKUMULASYON_DIP,FIB_ABSOLUTE_BOTTOM
253,BERA,72.80,ERKEN,68,100,SESSİZ_AKUMULASYON_DIP,FIB_ABSOLUTE_BOTTOM
262,SRVGY,71.95,ERKEN,67,100,SESSİZ_AKUMULASYON_DIP,FIB_ABSOLUTE_BOTTOM
17,HALKB,70.25,ERKEN,65,100,SESSİZ_AKUMULASYON_DIP,FIB_ABSOLUTE_BOTTOM
568,IHYAY,70.25,ERKEN,65,100,SESSİZ_AKUMULASYON_DIP,FIB_ABSOLUTE_BOTTOM
389,TKNSA,70.25,ERKEN,65,100,SESSİZ_AKUMULASYON_DIP,FIB_ABSOLUTE_BOTTOM
